In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 10 samples and 367 columns.
First few rows of the data:


,progression-free-survival_days,recurrence,overall-survival_days,survival,sample_title,tissue,tumor type,Sex,ISG15,TNFRSF18,...,TLR7,TLR8,CYBB,FOXP3,XAGE1A,XAGE1B,IL2RG,SH2D1A,CD40LG,G6PD
0,144,1,234,0,D01,lung,adenocarcinoma,female,8.882173,6.171445,...,5.284179,4.930614,8.576640,6.549989,8.074424,8.069476,7.577147,5.239719,5.509732,6.336172
1,1088,0,1088,0,D03,lung,adenocarcinoma,male,7.529397,5.248833,...,2.258170,3.079047,7.128186,3.958780,8.727017,8.725554,5.703426,3.967348,3.158742,6.494435
2,253,1,262,1,D06,lung,Large cell neuroendocarine carcinoma,male,8.251118,4.840282,...,5.837569,4.775874,8.362806,4.759907,0.000000,0.000000,7.162295,5.826552,3.886130,6.795918
3,259,1,259,1,D07,lung,adenocarcinoma,female,7.544768,4.033201,...,5.105420,3.636555,8.202394,4.115164,8.507414,8.501555,6.781775,5.287543,4.261330,6.541916
4,191,1,376,1,D08,lung,Large cell neuroendocarine carcinoma,male,7.112337,4.998744,...,6.330000,4.266616,8.244475,5.164976,7.329526,7.330164,6.668930,4.644956,3.693069,9.176531


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for progression-free-survival_days or recurrence: 0
Number of samples with complete data for both variables: 10
Number of censored samples (event_var == 0.0): 1
Number of events (event_var == 1.0): 9
Censored ratio (among complete): 1 / 10 = 0.100 (10.0%)
Number of samples with incomplete data for overall-survival_days or survival: 0
Number of samples with complete data for both variables: 10
Number of censored samples (event_var == 0.0): 2
Number of events (event_var == 1.0): 8
Censored ratio (among complete): 2 / 10 = 0.200 (20.0%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE136961/description.json
